## Step 1: Load Data

In [1]:
import numpy as np
import pandas as pd

In [2]:
import os
train_trans = pd.read_csv(os.path.join("..", "resources", "train_transaction.csv"))
train_id = pd.read_csv(os.path.join("..", "resources", "train_identity.csv"))
test_trans = pd.read_csv(os.path.join("..", "resources", "test_transaction.csv"))
test_id = pd.read_csv(os.path.join("..", "resources", "test_identity.csv"))

In [3]:
print(train_trans.shape)
print(train_id.shape)

(590540, 394)
(144233, 41)


In [4]:
train = train_trans.merge(train_id, on="TransactionID", how="left")
test = test_trans.merge(test_id, on="TransactionID", how="left")

In [5]:
del train_trans, train_id, test_trans, test_id

In [6]:
print(train.shape)
print(test.shape)

(590540, 434)
(506691, 433)


## Step 2: Analyze

In [7]:
fraud_rate = train['isFraud'].mean()
print(f"Fraud: {fraud_rate:.2%}")

Fraud: 3.50%


In [8]:
missing = train.isnull().sum() / len(train) * 100
missing_cols = missing[missing > 50].sort_values(ascending=False)

In [9]:
print(missing_cols)

id_24    99.196159
id_25    99.130965
id_07    99.127070
id_08    99.127070
id_21    99.126393
           ...    
M5       59.349409
M7       58.635317
M8       58.633115
M9       58.633115
D5       52.467403
Length: 214, dtype: float64


In [10]:
import re
# 批量重命名列：id-01 -> id_01
def rename_columns(df):
    # 正则匹配 id-01 到 id_01 格式
    new_columns = {}
    for col in df.columns:
        if re.match(r'^id-\d{2}$', col):
            new_col = col.replace('-', '_')
            new_columns[col] = new_col
    return df.rename(columns=new_columns)
# 应用到test和train
test = rename_columns(test)
train = rename_columns(train)

In [11]:
missing_cols = missing_cols.index.tolist()
train = train.drop(columns=missing_cols)
test = test.drop(columns=missing_cols)

In [12]:
print(train.shape)
print(test.shape)

(590540, 220)
(506691, 219)


In [14]:
numeric_cols = [c for c in train.select_dtypes(include=['number']).columns
                if c not in ['isFraud', 'TransactionID']]
train[numeric_cols] = train[numeric_cols].fillna(-999)
test[numeric_cols] = test[numeric_cols].fillna(-999)
categorical_cols = train.select_dtypes(include=['str']).columns.tolist()
train[categorical_cols] = train[categorical_cols].fillna('Unknown')
test[categorical_cols] = test[categorical_cols].fillna('Unknown')
print(f"数值特征: {len(numeric_cols)}")
print(f"类别特征: {len(categorical_cols)}")

数值特征: 209
类别特征: 9


In [15]:
print(categorical_cols)

['ProductCD', 'card4', 'card6', 'P_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M6']


In [17]:
from sklearn.preprocessing import LabelEncoder
# 对每个类别列编码
for col in categorical_cols:
    le = LabelEncoder()
    # 合并训练测试集编码，避免测试集出现新类别
    all_values = pd.concat([train[col], test[col]]).astype(str)
    le.fit(all_values)
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

In [18]:
train.to_csv(os.path.join("..", "resources", "train_preprocessed.csv"))
test.to_csv(os.path.join("..", "resources", "test_preprocessed.csv"))